In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster
import warnings
warnings.filterwarnings('ignore')

provider_risk = pd.read_csv('../outputs/provider_risk_scores.csv')
watchlist = pd.read_csv('../outputs/flagged_providers_watchlist.csv')

print(f"Provider risk scores loaded: {provider_risk.shape}")
print(f"Watchlist loaded: {watchlist.shape}")

Provider risk scores loaded: (62301, 18)
Watchlist loaded: (100, 18)


In [2]:
# City coordinates for major Texas cities
# This covers the majority of watchlist providers
texas_city_coords = {
    'Houston': (29.7604, -95.3698),
    'San Antonio': (29.4241, -98.4936),
    'Dallas': (32.7767, -96.7970),
    'Austin': (30.2672, -97.7431),
    'Fort Worth': (32.7555, -97.3308),
    'El Paso': (31.7619, -106.4850),
    'Arlington': (32.7357, -97.1081),
    'Corpus Christi': (27.8006, -97.3964),
    'Plano': (33.0198, -96.6989),
    'Lubbock': (33.5779, -101.8552),
    'Laredo': (27.5306, -99.4803),
    'Irving': (32.8140, -96.9489),
    'Garland': (32.9126, -96.6389),
    'Amarillo': (35.2220, -101.8313),
    'Grand Prairie': (32.7460, -96.9978),
    'Brownsville': (25.9017, -97.4975),
    'Pasadena': (29.6911, -95.2091),
    'Mesquite': (32.7668, -96.5992),
    'Mcallen': (26.2034, -98.2300),
    'Killeen': (31.1171, -97.7278),
    'Frisco': (33.1507, -96.8236),
    'Waco': (31.5493, -97.1467),
    'Carrollton': (32.9537, -96.8903),
    'Denton': (33.2148, -97.1331),
    'Midland': (31.9973, -102.0779),
    'Abilene': (32.4487, -99.7331),
    'Beaumont': (30.0860, -94.1018),
    'Round Rock': (30.5083, -97.6789),
    'Odessa': (31.8457, -102.3676),
    'Richardson': (32.9483, -96.7299),
    'Tyler': (32.3513, -95.3011),
    'Pearland': (29.5635, -95.2860),
    'Sugar Land': (29.6197, -95.6349),
    'Lewisville': (33.0462, -96.9942),
    'League City': (29.5074, -95.0949),
    'Edinburg': (26.3017, -98.1633),
    'Wichita Falls': (37.9716, -98.4928),
    'Allen': (33.1032, -96.6706),
    'San Angelo': (31.4638, -100.4370),
    'Longview': (32.5007, -94.7405)
}

# Match watchlist providers to coordinates
def get_coords(city):
    if pd.isna(city):
        return None
    city_title = str(city).title().strip()
    return texas_city_coords.get(city_title, None)

watchlist['coords'] = watchlist['Prscrbr_City'].apply(get_coords)
watchlist_mapped = watchlist[watchlist['coords'].notna()].copy()
watchlist_mapped['lat'] = watchlist_mapped['coords'].apply(lambda x: x[0])
watchlist_mapped['lon'] = watchlist_mapped['coords'].apply(lambda x: x[1])

print(f"Watchlist providers with coordinates: {len(watchlist_mapped)} of {len(watchlist)}")
print(f"\nCities represented:")
print(watchlist_mapped['Prscrbr_City'].value_counts().head(10))

Watchlist providers with coordinates: 87 of 100

Cities represented:
Prscrbr_City
Houston           34
Dallas            19
San Antonio       14
Fort Worth         4
Austin             3
Mcallen            2
El Paso            2
Laredo             2
Tyler              2
Corpus Christi     1
Name: count, dtype: int64


In [3]:
# Create base map centered on Texas
m = folium.Map(
    location=[31.0, -99.0],
    zoom_start=6,
    tiles='CartoDB positron'
)

# Add marker cluster for clean visualization
marker_cluster = MarkerCluster(name='Flagged Providers').add_to(m)

# Color function based on risk score
def get_color(risk_score):
    if risk_score >= 10:
        return '#d32f2f'  # Critical — red
    elif risk_score >= 5:
        return '#f57c00'  # High — orange
    else:
        return '#fbc02d'  # Moderate — yellow

# Add a marker for each flagged provider
for _, row in watchlist_mapped.iterrows():
    color = get_color(row['risk_score'])
    
    popup_html = f"""
    <div style='font-family: Arial; width: 220px;'>
        <h4 style='color: {color}; margin-bottom: 5px;'>
            ⚠️ {row['Prscrbr_Last_Org_Name']}, {row['Prscrbr_First_Name']}
        </h4>
        <b>Specialty:</b> {row['Prscrbr_Type']}<br>
        <b>City:</b> {row['Prscrbr_City']}<br>
        <b>NPI:</b> {row['Prscrbr_NPI']}<br>
        <hr>
        <b>Risk Score:</b> {row['risk_score']:.2f}<br>
        <b>Total Cost:</b> ${row['total_cost']:,.2f}<br>
        <b>Total Claims:</b> {row['total_claims']:,}<br>
        <b>Unique Drugs:</b> {row['unique_drugs']}<br>
        <b>Total Flags:</b> {row['total_flags']}
    </div>
    """
    
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=8 + (row['risk_score'] * 0.5),
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"{row['Prscrbr_Last_Org_Name']} | Risk: {row['risk_score']:.2f}"
    ).add_to(marker_cluster)

# Add legend
legend_html = """
<div style='position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background-color: white; padding: 15px; border-radius: 8px;
     border: 2px solid #ccc; font-family: Arial; font-size: 13px;'>
    <b>Provider Risk Level</b><br><br>
    <span style='color: #d32f2f;'>●</span> Critical Risk (10+)<br>
    <span style='color: #f57c00;'>●</span> High Risk (5-10)<br>
    <span style='color: #fbc02d;'>●</span> Moderate Risk (2-5)<br><br>
    <small>Circle size = risk score magnitude</small>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Save map
m.save('../visualizations/maps/texas_fraud_risk_map.html')
print("Interactive map saved")

Interactive map saved


In [4]:
print("=" * 55)
print("   MEDICARE PART D FRAUD DETECTION: FINAL SUMMARY")
print("   Texas 2023")
print("=" * 55)

print(f"""
DATASET
  Total records analyzed:        {869717:,}
  Unique providers analyzed:     {62301:,}
  Unique drugs analyzed:         {1449:,}
  State:                         Texas

FLAGGED PROVIDERS
  Total flagged (watchlist):     {len(watchlist):,}
  Flagged rate:                  {len(watchlist)/62301*100:.2f}%
  Total suspicious spend:        ${watchlist['total_cost'].sum():,.2f}
  Average risk score:            {watchlist['risk_score'].mean():.2f}
  Highest risk score:            {watchlist['risk_score'].max():.2f}

RISK TIERS
  Critical Risk (10+):           {len(watchlist[watchlist['risk_score'] >= 10]):,} providers
  High Risk (5-10):              {len(watchlist[(watchlist['risk_score'] >= 5) & (watchlist['risk_score'] < 10)]):,} providers
  Moderate Risk (2-5):           {len(watchlist[watchlist['risk_score'] < 5]):,} providers

KEY FINDINGS
  Specialty mismatches found:    16 ophthalmologists prescribing opioids
  Highest single provider cost:  $50,570,752.74 (Internal Medicine)
  Most flagged specialty:        Nurse Practitioners
""")
print("=" * 55)

   MEDICARE PART D FRAUD DETECTION: FINAL SUMMARY
   Texas 2023

DATASET
  Total records analyzed:        869,717
  Unique providers analyzed:     62,301
  Unique drugs analyzed:         1,449
  State:                         Texas

FLAGGED PROVIDERS
  Total flagged (watchlist):     100
  Flagged rate:                  0.16%
  Total suspicious spend:        $274,521,439.65
  Average risk score:            5.12
  Highest risk score:            24.06

RISK TIERS
  Critical Risk (10+):           6 providers
  High Risk (5-10):              26 providers
  Moderate Risk (2-5):           68 providers

KEY FINDINGS
  Specialty mismatches found:    16 ophthalmologists prescribing opioids
  Highest single provider cost:  $50,570,752.74 (Internal Medicine)
  Most flagged specialty:        Nurse Practitioners

